# Clustering

In [ ]:
#general packages
import pandas as pd
import numpy as np
import scanpy as sc
from glob import glob
import matplotlib.pyplot as plt
import squidpy as sq
%config InlineBackend.figure_format='retina'

import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

In [ ]:
from skimage.measure import regionprops
import tifffile as tf
import json
from pathlib import Path

def convert_pos_file_to_csv(filename: str, px_size: float):
        """
        Parses stage positions from a JSON `.pos` file and converts them to a CSV format.

        :param img_path: Path to the directory containing the `.pos` file.
        :param output_file_name: Optional output file name for the stitched image.
        :return: A DataFrame with position information and the output file path.
        """

        with open(filename) as f:
            content = json.load(f)

        positions = []
        for pos in content["map"]['StagePositions']["array"]:
            label = pos["Label"]["scalar"]
            cor = pos['DevicePositions']["array"]

            index = 1 if cor[0]["Device"]["scalar"] == 'Adaptive Focus Control Offset' else 0
            coordinates = cor[index]['Position_um']["array"]

            if len(coordinates) < 2:
                print(f"Warning: Position {label} has incomplete coordinates. Skipping.")
                continue

            posinfo = {
                'label': label,
                'x': coordinates[0],
                'xpx': round(coordinates[0] / px_size),
                'y': coordinates[1],
                'ypx': round(coordinates[1] / px_size)
            }
            positions.append(posinfo)
        df = pd.DataFrame(positions)
        df["label"] = [f"Pos{pos}" for pos in range(0,len(df))]
        return df
    
def getCellLoc(src, pos_df):
    #get information about masks
    pos_info = Path(src).name.split("_")[1]
    mask = tf.imread(src)
    regions = regionprops(mask)
    finaldict = dict()
    for props in regions:
        y0, x0 = props.centroid
        # Now need to identify the proper offset
        subsetpositionfile = pos_df[pos_df['label'] == pos_info]
        x_final = x0 + subsetpositionfile['xpx'].values[0]
        y_final = y0 + subsetpositionfile['ypx'].values[0]
        fullkey = 'Cell' + str(props.label) + '.0_' + pos_info[:3] + f"_{pos_info[3:]}"
        finaldict[fullkey] = [x0, y0, x_final, y_final, pos_info]
    return finaldict

In [ ]:
#create df from positions file
pos_df = convert_pos_file_to_csv("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pos.pos", 0.103)

#read in all masks
srcs = glob("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/edges_deleted/*_z0.tif")

#fill dictionary 
newdict = dict()
for src in srcs: 
    tempdict = getCellLoc(src, pos_df)
    newdict.update(tempdict)

In [ ]:
#transpose and rename columns
CellLocationsAndPositions = pd.DataFrame(newdict).T
CellLocationsAndPositions = CellLocationsAndPositions.rename(columns = {0: "x_rel", 1: "y_rel", 2: "x", 3: "y", 4: "Pos"})